In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
!pip install timm
!pip install torch torchvision

In [3]:
from google.colab import files

files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"venkatasaimohit","key":"77968d38d0966579c078f3dbe053fee2"}'}

In [4]:
!mkdir -p ~/.kaggle
!cp "kaggle (2).json" ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle (2).json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [5]:
!kaggle datasets download -d dansbecker/food-101

Dataset URL: https://www.kaggle.com/datasets/dansbecker/food-101
License(s): other
100% 9.38G/9.38G [07:23<00:00, 22.7MB/s]



In [6]:
!unzip -q food-101.zip

replace food-101.zip? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
y


In [7]:
import os
print(os.listdir("/content"))

['.config', 'food-101', 'food-101.zip', 'kaggle.json', 'sample_data']


In [8]:
import os
print(os.listdir("/content/food-101"))

['food-101', '__MACOSX']


In [9]:
import os
print(len(os.listdir("/content/food-101/food-101/images")))

102


In [10]:
from torchvision import datasets
from torchvision import transforms

In [11]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [12]:
dataset = datasets.ImageFolder(
    root="/content/food-101/food-101/images",
    transform=transform
)

In [13]:
print(len(dataset))
print(len(dataset.classes))

101000
101


In [14]:
from torch.utils.data import random_split

In [15]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(
    dataset,
    [train_size, val_size]
)

In [16]:
from torch.utils.data import DataLoader

In [17]:
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [18]:
import timm

In [19]:
model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=101
)

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [20]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

In [21]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

In [22]:
import torch.optim as optim

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [ ]:
epochs = 5

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs} Loss: {running_loss/len(train_loader):.4f}"
    )

Epoch 1/5 Loss: 1.7383


In [ ]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

accuracy = 100 * correct / total

print(
    f"Validation Accuracy: {accuracy:.2f}%"
)

In [ ]:
torch.save(
    model.state_dict(),
    "food_classifier.pth"
)

In [ ]:
from google.colab import files

files.download(
    "food_classifier.pth"
)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from PIL import Image

img_path = list(uploaded.keys())[0]

img = Image.open(
    img_path
).convert("RGB")

In [ ]:
image = transform(img)

image = image.unsqueeze(0)

image = image.to(device)

NameError: name 'device' is not defined

In [ ]:
model.eval()

with torch.no_grad():

    output = model(image)

    probs = torch.softmax(output, dim=1)

    top_probs, top_idxs = torch.topk(
        probs,
        5
    )

for p, idx in zip(top_probs[0], top_idxs[0]):
    print(
        dataset.classes[idx.item()],
        float(p)
    )

In [ ]:
torch.save(
    model.state_dict(),
    "food_classifier_77.pth"
)

In [ ]:
from google.colab import files

files.download("food_classifier_77.pth")